In [ ]:
!nvidia-smi | head -12
%cd /content
!rm -rf ConInfer
!git clone -q https://github.com/Dog-Yang/ConInfer.git
%cd /content/ConInfer
!mkdir -p data/loveDA
!nohup wget -q -c https://zenodo.org/record/5706578/files/Val.zip -O data/loveDA/Val.zip > wget.log 2>&1 &
!pip install -q torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126
!grep -v -E "^(mmcv|pydensecrf|pykeops|torch)" requirements.txt > req_light.txt
!pip install -q -r req_light.txt
!pip install -q mmcv-lite==2.1.0 pykeops==2.3 ftfy regex prettytable torchmetrics submitit termcolor

Mon Oct  5 05:31:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import mmcv, os
stub = os.path.join(os.path.dirname(mmcv.__file__), "_ext.py")
open(stub, "w").write('''
def __getattr__(name):
    if name.startswith("__"):
        raise AttributeError(name)
    def _missing(*a, **k):
        raise NotImplementedError(f"mmcv compiled op {name} is not available here")
    return _missing
''')
!git clone -q https://github.com/likyoo/SegEarth-OV.git /content/SegEarth-OV
!cp /content/SegEarth-OV/segearth_segmentor.py /content/ConInfer/
!git clone -q https://github.com/facebookresearch/dinov3.git /content/dinov3-main
from huggingface_hub import hf_hub_download
p = hf_hub_download("MVRL/dinov3_vitl16_sat", "dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth", local_dir="/content/weights")
!sha256sum {p} | cut -c1-8
%cd /content/ConInfer
!sed -i 's|/data/users/cwy/0_datasets/github_source_packge/dinov3-main|/content/dinov3-main|g; s|/data/users/cwy/0_datasets/model_weight/DINOV3|/content/weights|g' ConInfer_segmentor.py
!sed -i 's/num_workers=32/num_workers=2/' configs_ConInfer/cfg_loveda.py
!grep -n "batch_size\|num_workers" configs_ConInfer/cfg_loveda.py

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dinov3_vitl16_pretrain_sat493m-eadcf0ff.(…):   0%|          | 0.00/1.21G [00:00<?, ?B/s]

eadcf0ff
/content/ConInfer
26:    batch_size=50,
27:    num_workers=2,


In [ ]:
%cd /content/ConInfer
!while pgrep -x wget > /dev/null; do sleep 10; done; ls -lh data/loveDA
!sed -i "s/for dataset in \['Train', 'Val'\]:/for dataset in ['Val']:/" tools/dataset_converters/loveda.py
!python tools/dataset_converters/loveda.py data/loveDA --out_dir data/LoveDA_val
!ln -sfn /content/ConInfer/data/LoveDA_val /content/ConInfer/data/LoveDA
!ls data/LoveDA/img_dir/val | wc -l
!python eval.py --config configs_ConInfer/cfg_loveda.py --work-dir ./output_coninfer_b50 --launcher none 2>&1 | tee run_coninfer_b50.log | tail -25

/content/ConInfer
total 2.3G
-rw-r--r-- 1 root root 2.3G Oct  5 05:44 Val.zip
Making directories...
Removing the temporary files...
Done!
1669
 -------------------- 
after_run:
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
10/05 05:51:35 - mmengine - WARNING - The prefix is not set in metric class IoUMetric.
10/05 05:52:21 - mmengine - INFO - Iter(test) [ 5/34]    eta: 0:04:25  time: 9.1640  data_time: 1.1433  memory: 8258  
10/05 05:52:56 - mmengine - INFO - Iter(test) [10/34]    eta: 0:03:13  time: 8.0589  data_time: 0.6766  memory: 8257  
10/05 05:53:31 - mmengine - INFO - Iter(test) [15/34]    eta: 0:02:26  time: 6.9827  data_time: 0.2176  memory: 8257  
10/05 05:54:07 - mmengine - INFO - Iter(test) [20/34]    eta: 0:01:46  time: 7.1526  data_time: 0.1997  memory: 8257  
10/05 05:54:44 - mmengine - INFO - Iter(test) [25/34]    eta: 0:01:07  time: 7.2913  data_time: 0.1995  memory: 8257  
10/05 05:55:21 - mmengine - INFO - Iter(test) [30/34]    eta: 0:00:

In [ ]:
!zip -rq coninfer_b50.zip output_coninfer_b50 run_coninfer_b50.log
from google.colab import files
files.download('coninfer_b50.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%cd /content/ConInfer

/content/ConInfer


In [ ]:
%%bash
cd /content/ConInfer
CFG=configs_ConInfer/cfg_loveda.py
for B in 8 25 50; do
  echo "=============== batch_size=$B ==============="
  sed -i -E "s/batch_size *= *[0-9]+/batch_size=$B/" $CFG
  grep -n "batch_size" $CFG
  python eval.py --config $CFG --work-dir ./output_coninfer_b$B --launcher none > run_coninfer_b$B.log 2>&1
  echo "--- result for batch $B ---"
  grep "aAcc" run_coninfer_b$B.log | tail -1
done
sed -i -E "s/batch_size *= *[0-9]+/batch_size=50/" $CFG

=============== batch_size=8 ===============
26:    batch_size=8,
--- result for batch 8 ---
10/05 06:51:19 - mmengine - INFO - Iter(test) [209/209]    aAcc: 52.8600  mIoU: 36.9900  mAcc: 59.8200  data_time: 0.0343  time: 1.1608
=============== batch_size=25 ===============
26:    batch_size=25,
--- result for batch 25 ---
10/05 06:55:58 - mmengine - INFO - Iter(test) [67/67]    aAcc: 55.4300  mIoU: 39.0200  mAcc: 60.8700  data_time: 0.1211  time: 3.7436
=============== batch_size=50 ===============
26:    batch_size=50,
--- result for batch 50 ---
10/05 07:00:32 - mmengine - INFO - Iter(test) [34/34]    aAcc: 56.1500  mIoU: 39.7800  mAcc: 61.3800  data_time: 0.2923  time: 7.2762


In [ ]:
import re
for B in [8, 25, 50]:
    txt = open(f"/content/ConInfer/run_coninfer_b{B}.log").read()
    m = re.findall(r"aAcc: ([\d.]+)\s+mIoU: ([\d.]+)\s+mAcc: ([\d.]+)", txt)
    print(B, m[-1] if m else "no result found")

8 ('52.8600', '36.9900', '59.8200')
25 ('55.4300', '39.0200', '60.8700')
50 ('56.1500', '39.7800', '61.3800')


In [ ]:
!cd /content/ConInfer && zip -r /content/coninfer_results.zip run_coninfer_b8.log run_coninfer_b25.log run_coninfer_b50.log configs_ConInfer/cfg_loveda.py ConInfer_segmentor.py
from google.colab import files
files.download('/content/coninfer_results.zip')